# Hierarchical (Agglomerative) Clustering

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## What Makes Hierarchical Clustering Different

K-Means and DBSCAN both produce a **flat partition** — one set of cluster labels at one level of granularity. Hierarchical Clustering produces something richer:

> *A tree of all possible clusterings at every level of similarity.*

You don't commit to K upfront. You build the full tree, visualize it as a **dendrogram**, and then decide where to cut.

---

## The Algorithm

**Agglomerative** (bottom-up):
1. Start: every point is its own cluster (n clusters)
2. Find the two closest clusters and merge them
3. Repeat until one cluster remains
4. The full merge history is the dendrogram
5. Cut at any height to extract any number of clusters

### Linkage — Distance Between Clusters

| Linkage | Distance = | Behavior |
|---|---|---|
| Single | Min pairwise distance | Chain-like clusters |
| Complete | Max pairwise distance | Compact clusters |
| Average | Mean pairwise distance | Balanced |
| Ward | Increase in within-cluster variance | Most like K-Means |

### Reading the Dendrogram

- **Height** of a merge = how dissimilar the two clusters were when they joined
- **Large gaps** between merge heights suggest natural cluster boundaries
- **Cutting** at a given height extracts clusters at that granularity


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram

import sys
sys.path.insert(0, '../../src')
from football_ml.unsupervised_learning.hierarchical import HierarchicalClustering

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Toy Example — Dendrogram Intuition

In [ ]:
rng = np.random.default_rng(0)
X0 = rng.normal(loc=[0, 0],   scale=0.3, size=(10, 2))
X1 = rng.normal(loc=[5, 0],   scale=0.3, size=(10, 2))
X2 = rng.normal(loc=[2.5, 4], scale=0.3, size=(10, 2))
X_toy = np.vstack([X0, X1, X2])

hc_toy = HierarchicalClustering(n_clusters=3, linkage='ward').fit(X_toy)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter coloured by cluster
colors = ['#4878CF', '#E87272', '#2ecc71']
for j in range(3):
    mask = hc_toy.labels_ == j
    axes[0].scatter(X_toy[mask, 0], X_toy[mask, 1],
                    color=colors[j], s=60, edgecolors='white',
                    linewidth=0.5, label=f'Cluster {j}')
axes[0].set_title('Clustering Result (n_clusters=3)', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)

# Dendrogram using our linkage matrix + scipy's renderer
Z = hc_toy.get_dendrogram_data()
dendrogram(Z, ax=axes[1], color_threshold=0,
           above_threshold_color='gray', no_labels=True)
axes[1].set_xlabel('Data points', fontsize=11)
axes[1].set_ylabel('Merge distance', fontsize=11)
axes[1].set_title('Dendrogram — Full Merge History', fontsize=11, fontweight='bold')
# Mark where we cut for 3 clusters
cut_height = (Z[-3, 2] + Z[-2, 2]) / 2
axes[1].axhline(cut_height, color='black', linestyle='--', linewidth=1.5,
                label=f'Cut for 3 clusters')
axes[1].legend(fontsize=9)

plt.suptitle('Hierarchical Clustering — Toy Example', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

The dendrogram reads bottom-up. Leaves are individual points. Each horizontal bar is a merge. The height of the bar shows how different the two groups were when they merged. The dashed cut line shows where we extract 3 clusters — everything below the cut is in the same cluster.

---
## 2. Linkage Comparison

In [ ]:
from sklearn.datasets import make_moons
X_moons, _ = make_moons(n_samples=60, noise=0.08, random_state=SEED)

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, linkage in zip(axes, ['single', 'complete', 'average', 'ward']):
    m = HierarchicalClustering(n_clusters=2, linkage=linkage).fit(X_moons)
    for j in range(2):
        mask = m.labels_ == j
        ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                   color=['#4878CF','#E87272'][j], s=20, alpha=0.8)
    ax.set_title(f'{linkage.capitalize()} linkage', fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Effect of Linkage Method on Moon Data', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

Single linkage's chaining tendency makes it correctly follow the moon shapes. Complete and Ward impose more compact, spherical clusters. Average is a compromise. Linkage choice matters significantly for non-spherical data.

---
## 3. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['home_win']    = (df['home_score'] > df['away_score']).astype(int)
df['goal_diff']   = df['home_score'] - df['away_score']
df['total_goals'] = df['home_score'] + df['away_score']

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['home_win', 'goal_diff', 'total_goals']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches')

---
## 4. Prepare Data — Small Sample

In [ ]:
# Hierarchical clustering is O(n²) — use a sample
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(df_clean), size=500, replace=False)
df_sample  = df_clean.iloc[sample_idx].copy()

X = df_sample[feature_cols].values
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

print(f'Using {len(X_sc)} samples')

---
## 5. Full Dendrogram

In [ ]:
hc_full = HierarchicalClustering(n_clusters=3, linkage='ward').fit(X_sc)
Z = hc_full.get_dendrogram_data()

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, ax=ax, truncate_mode='level', p=5,
           color_threshold=0, above_threshold_color='#4878CF',
           no_labels=True)
ax.set_xlabel('Matches (merged)', fontsize=12)
ax.set_ylabel('Ward distance', fontsize=12)
ax.set_title('Football Matches — Hierarchical Clustering Dendrogram (Ward)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

The dendrogram is truncated to the top 5 levels for readability. The large gaps near the top indicate the natural cluster boundaries — cutting at the biggest gap gives the most meaningful partition.

---
## 6. Choosing the Number of Clusters — Gap Analysis

In [ ]:
# Last merges have the largest distances — gaps between them suggest K
last_merges = Z[-15:, 2][::-1]   # last 15 merge distances, reversed
gaps = np.diff(last_merges)
suggested_k = np.argmax(gaps) + 2   # +2 because we're looking at gaps

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(2, len(last_merges) + 1), last_merges[1:],
        'o-', color='#4878CF', linewidth=2)
ax.axvline(suggested_k, color='#E87272', linestyle='--',
           linewidth=1.5, label=f'Suggested K = {suggested_k}')
ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Merge Distance', fontsize=12)
ax.set_title('Gap Analysis — Finding Natural K', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
print(f'Suggested number of clusters: {suggested_k}')

---
## 7. Extract Clusters and Profile

In [ ]:
best_k = suggested_k
hc = HierarchicalClustering(n_clusters=best_k, linkage='ward').fit(X_sc)

df_sample = df_sample.copy()
df_sample['cluster'] = hc.labels_

print(f'Cluster sizes: {dict(zip(*np.unique(hc.labels_, return_counts=True)))}')
print()

profile = df_sample.groupby('cluster')[['goal_diff', 'home_win', 'total_goals']].mean()
print('Cluster profiles:')
print(profile.round(3).to_string())

In [ ]:
# Heatmap of cluster feature means
feat_profile = df_sample.groupby('cluster')[feature_cols].mean()
feat_profile_sc = pd.DataFrame(
    StandardScaler().fit_transform(feat_profile),
    index=feat_profile.index,
    columns=feature_cols
)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(feat_profile_sc, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, annot_kws={'size': 8})
ax.set_title('Cluster Feature Profiles (standardized means)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Comparison of All Unsupervised Algorithms

In [ ]:
from football_ml.unsupervised_learning.kmeans import KMeans
from football_ml.unsupervised_learning.pca import PCA

# Project to 2D with PCA for visualization
pca = PCA(n_components=2).fit(X_sc)
X_2d = pca.transform(X_sc)

# Fit all clustering algorithms
km = KMeans(k=best_k, n_init=5, random_state=SEED).fit(X_sc)
hc_plot = HierarchicalClustering(n_clusters=best_k, linkage='ward').fit(X_sc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cmap = plt.cm.tab10

for ax, labels, title in zip(
    axes,
    [km.labels_, hc_plot.labels_],
    ['K-Means', 'Hierarchical (Ward)']
):
    for j in range(best_k):
        mask = labels == j
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   color=cmap(j / best_k), s=12, alpha=0.6, label=f'Cluster {j}')
    ax.set_title(f'{title} (K={best_k})', fontsize=11, fontweight='bold')
    ax.set_xlabel('PC1', fontsize=10)
    ax.set_ylabel('PC2', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Clustering Comparison — PCA 2D Projection', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Discussion

### What Hierarchical Clustering does well
- **No K upfront**: the dendrogram lets you explore all possible granularities and decide after seeing the data
- **Deterministic**: unlike K-Means, no random initialization — same data always gives the same tree
- **Interpretable**: the dendrogram is a readable artifact showing the full cluster structure
- **Flexible linkage**: Ward behaves like K-Means, single linkage follows arbitrary shapes

### What it struggles with
1. **O(n²) complexity**: pairwise distances for large datasets are expensive — we used 500 samples
2. **Greedy merges**: once two points are merged, they can never be separated. Bad early merges propagate.
3. **Linkage sensitivity**: different linkage methods can give very different results on the same data
4. **No noise handling**: unlike DBSCAN, every point gets assigned to a cluster regardless

### Unsupervised learning recap

| Algorithm | Goal | K needed | Noise | Shape |
|---|---|---|---|---|
| K-Means | Find K clusters | Yes | No | Spherical |
| DBSCAN | Find dense regions | No | Yes (-1) | Any |
| PCA | Reduce dimensions | n/a | n/a | Linear |
| Hierarchical | Build cluster tree | No (choose after) | No | Depends on linkage |

---
## Summary

| | |
|---|---|
| **Algorithm type** | Agglomerative hierarchical clustering |
| **Approach** | Bottom-up merging of closest clusters |
| **Output** | Dendrogram + flat labels at chosen K |
| **K required** | No — chosen after viewing dendrogram |
| **Linkage options** | Single, Complete, Average, Ward |
| **Complexity** | O(n²) — slow on large data |
| **Key advantage** | Full tree of all clusterings, deterministic |
| **Key limitation** | Greedy merges are irreversible, no noise handling |
